In [1]:

import pandas as pd 
from biomart import BiomartServer
from io import StringIO


In [2]:
input_path = "df_all_tx_micro.csv"
dataset = pd.read_csv(input_path, low_memory=False)
print (dataset.shape)

(77, 61250)


In [3]:
server = BiomartServer("http://www.ensembl.org/biomart")

# Dataset humano 
ds = server.datasets["hsapiens_gene_ensembl"]

In [24]:
#ds.show_filters()
#ds.show_attributes()

In [25]:
all_attrs = list(ds.attributes.keys()) 
# print(all_attrs)

In [6]:
# escogi transctipt_biotype y de alli protein_coding (debería incluir otro?)
#query 

query = {
    "filters": {
        "transcript_biotype": "protein_coding"
    },
    "attributes": [
        "ensembl_gene_id",
        "external_gene_name",
        "ensembl_transcript_id",
        "transcript_biotype", 
        "hgnc_symbol"
    ]
}


In [7]:
response = ds.search(query, header=1)

tsv = "\n".join(line.decode("utf-8") for line in response.iter_lines() if line)
df = pd.read_csv(StringIO(tsv), sep="\t")
df.head()


,Gene stable ID,Gene name,Transcript stable ID,Transcript type,HGNC symbol
0,ENSG00000198888,MT-ND1,ENST00000361390,protein_coding,MT-ND1
1,ENSG00000198763,MT-ND2,ENST00000361453,protein_coding,MT-ND2
2,ENSG00000198804,MT-CO1,ENST00000361624,protein_coding,MT-CO1
3,ENSG00000198712,MT-CO2,ENST00000361739,protein_coding,MT-CO2
4,ENSG00000228253,MT-ATP8,ENST00000361851,protein_coding,MT-ATP8


In [8]:
df.shape

(221639, 5)

In [28]:
query2 = {
    "filters": {
        "hgnc_symbol": "LRFN3"
    },
    "attributes": [
        "hgnc_symbol",
        "ensembl_gene_id",
        "external_gene_name",
        "ensembl_transcript_id",
        "transcript_biotype" 
       
    ]
}

In [29]:
response = ds.search(query2, header=1)

tsv2 = "\n".join(line.decode("utf-8") for line in response.iter_lines() if line)
df2 = pd.read_csv(StringIO(tsv2), sep="\t")
df2.head()

,HGNC symbol,Gene stable ID,Gene name,Transcript stable ID,Transcript type
0,LRFN3,ENSG00000126243,LRFN3,ENST00000585876,protein_coding_CDS_not_defined
1,LRFN3,ENSG00000126243,LRFN3,ENST00000588831,protein_coding
2,LRFN3,ENSG00000126243,LRFN3,ENST00000587257,protein_coding_CDS_not_defined
3,LRFN3,ENSG00000126243,LRFN3,ENST00000246529,protein_coding


In [11]:
df2.shape
#salen repetidos

(10, 5)

In [12]:
#siguiente paso 
#de la lista df_all_tx_micro o como se llama, tomar todos los headers y 
#limpiar solo los que inicien con tx__ quitarles el 
#tx___ y esos buscarlos en el search query este

In [13]:
#si, encontrado - se queda esa columna en df_tx_all_filtrado, si no encontrado, esa columna se va

In [14]:
cols = pd.Index(dataset.columns)

tx_cols = cols[cols.str.startswith("tx__")]
symbols = (
    tx_cols.to_series(index=None)
          .str.replace(r"^tx__", "", regex=True)
          .str.strip()
          .tolist()
)

print("Columns:", len(cols))
print("Columns tx__:", len(tx_cols))


Columns: 61250
Columns tx__: 55765


In [15]:

mart = pd.read_csv(StringIO(tsv), sep="\t")
mart.head()


,Gene stable ID,Gene name,Transcript stable ID,Transcript type,HGNC symbol
0,ENSG00000198888,MT-ND1,ENST00000361390,protein_coding,MT-ND1
1,ENSG00000198763,MT-ND2,ENST00000361453,protein_coding,MT-ND2
2,ENSG00000198804,MT-CO1,ENST00000361624,protein_coding,MT-CO1
3,ENSG00000198712,MT-CO2,ENST00000361739,protein_coding,MT-CO2
4,ENSG00000228253,MT-ATP8,ENST00000361851,protein_coding,MT-ATP8


In [17]:
mart.shape

(221639, 5)

In [18]:
coding_symbols = (
    mart["HGNC symbol"]
    .dropna()
    .astype(str)
    .str.strip()
)

coding_set = set(coding_symbols)

In [19]:
tx_cols = [c for c in dataset.columns if isinstance(c, str) and c.startswith("tx__")]
non_tx_cols = [c for c in dataset.columns if c not in tx_cols]

In [20]:
tx_cols_keep = []
tx_cols_drop = []

for c in tx_cols:
    symbol = c.replace("tx__", "", 1).strip()   # quita solo el prefijo una vez
    if symbol in coding_set:
        tx_cols_keep.append(c)
    else:
        tx_cols_drop.append(c)

In [21]:
dataset_code = dataset[non_tx_cols + tx_cols_keep]

In [22]:
print("Columnas totales:", dataset.shape[1])
print("tx__ totales:", len(tx_cols))
print("tx__ que se quedan (protein_coding en mart):", len(tx_cols_keep))
print("tx__ que se van (no encontradas / raras / no codificantes):", len(tx_cols_drop))
print("Columnas finales:", dataset_code.shape[1])

print("\nEjemplos de tx__ eliminadas (primeras 30):")
print(tx_cols_drop[:30])

Columnas totales: 61250
tx__ totales: 55765
tx__ que se quedan (protein_coding en mart): 17732
tx__ que se van (no encontradas / raras / no codificantes): 38033
Columnas finales: 23217

Ejemplos de tx__ eliminadas (primeras 30):
['tx__5S_rRNA', 'tx__7SK', 'tx__A1BG-AS1', 'tx__A2M-AS1', 'tx__A2ML1-AS1', 'tx__A2ML1-AS2', 'tx__A2MP1', 'tx__AACSP1', 'tx__AAED1', 'tx__AARS', 'tx__AATK-AS1', 'tx__AB015752.3', 'tx__AB019438.66', 'tx__AB019440.1', 'tx__AB019440.50', 'tx__AB019441.29', 'tx__ABC12-47043100G14.2', 'tx__ABC7-42389800N19.1', 'tx__ABC7-43041300I9.1', 'tx__ABCA11P', 'tx__ABCA17P', 'tx__ABCA9-AS1', 'tx__ABCB10P1', 'tx__ABCB10P3', 'tx__ABCB10P4', 'tx__ABCC13', 'tx__ABCC5-AS1', 'tx__ABCC6P1', 'tx__ABCC6P2', 'tx__ABCD1P2']


In [23]:
dataset_code.to_csv("df_all_tx_micro_PROTEIN_CODING_ONLY.csv", index=False)